In [ ]:
# Stage 04 - data acquisition and ingestion

In [1]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..')  # project/notebooks -> project

ROOT = Path.cwd()

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('working from:', ROOT.name)

working from: project


In [2]:
import os
import datetime as dt
import pathlib

from typing import Dict, List

import requests
import pandas as pd
from dotenv import load_dotenv

In [6]:
DATA_RAW = pathlib.Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

load_dotenv()

ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")

print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))
DATA_RAW.mkdir(parents=True, exist_ok=True)
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
bool(ALPHA_KEY)

Loaded ALPHAVANTAGE_API_KEY? False


False

In [9]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    dt.datetime.now()
    strftime("%Y%m%d-%H%M%S")

In [10]:
def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join(
        [f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()]
    )
    return f"{prefix}_{mid}_{safe_stamp()}.csv"

In [11]:
def validate_df(
    df: pd.DataFrame,
    required_cols: List[str],
    dtypes_map: Dict[str, str]
) -> Dict[str, str]:

    msgs = {}

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"

    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = (
                    f"Failed to coerce {col} to {dtype}: {e}"
                )

    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"

    return msgs

In [13]:
SYMBOL = "SPY"
use_alpha = bool(ALPHA_KEY)

print("Using Alpha Vantage:", use_alpha)
print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }

    r = requests.get(
        url,
        params=params,
        timeout=30
    )

    r.raise_for_status()

    js = r.json()

    key = [
        k for k in js.keys()
        if "Time Series" in k
    ]

    if not key:
        print(
            "Alpha Vantage returned no series:",
            str(list(js.values())[0])[:150]
        )
        use_alpha = False

Using Alpha Vantage: False
Using Alpha Vantage: False


In [ ]:
if use_alpha:

    series = js[key[0]]

    df_api = (
        pd.DataFrame(series)
        .T
        .rename_axis("date")
        .reset_index()
    )

    df_api = (
        df_api[["date", "4. close"]]
        .rename(columns={"4. close": "close"})
    )

    df_api["date"] = pd.to_datetime(
        df_api["date"]
    )

    df_api["close"] = pd.to_numeric(
        df_api["close"]
    )

In [15]:
if not use_alpha:

    import yfinance as yf

    df_api = (
        yf.download(
            SYMBOL,
            period="10y",
            interval="1d",
            auto_adjust=False,
            multi_level_index=False
        )
        .reset_index()[["Date", "Close"]]
    )

    df_api.columns = ["date", "close"]

[*********************100%***********************]  1 of 1 completed


In [17]:
df_api = (
    df_api
    .sort_values("date")
    .reset_index(drop=True)
)

In [ ]:
msgs = validate_df(
    df_api,
    required_cols=["date", "close"],
    dtypes_map={
        "date": "datetime64[ns]",
        "close": "float"
    }
)

print(msgs)

In [18]:
fname = safe_filename(
    prefix="api",
    meta={
        "source": "alpha" if use_alpha else "yfinance",
        "symbol": SYMBOL
    }
)

out_path = DATA_RAW / fname

df_api.to_csv(
    out_path,
    index=False
)

print("Saved:", out_path)

Saved: data\raw\api_source-yfinance_symbol-SPY_20260825-160153.csv


In [ ]:
# Stage 05 — Data Storage

In [21]:
import os
import pathlib
import pandas as pd
from dotenv import load_dotenv

In [22]:
load_dotenv()

RAW_DIR = pathlib.Path(
    os.getenv("DATA_DIR_RAW", "data/raw")
)

PROC_DIR = pathlib.Path(
    os.getenv("DATA_DIR_PROCESSED", "data/processed")
)

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())

RAW_DIR: C:\Users\PC\bootcamp_chen_he\project\data\raw
PROC_DIR: C:\Users\PC\bootcamp_chen_he\project\data\processed


In [25]:
spy_files = sorted(
    RAW_DIR.glob("*SPY*.csv")
)

print("SPY raw files found:", len(spy_files))
latest_spy_path = spy_files[-1]

print("Loading:", latest_spy_path)

SPY raw files found: 1
Loading: data\raw\api_source-yfinance_symbol-SPY_20260825-160153.csv


In [26]:
spy_raw = pd.read_csv(
    latest_spy_path,
    parse_dates=["date"]
)

In [27]:
def validate_spy_storage(df):
    checks = {
        "not_empty": not df.empty,
        "date_present": "date" in df.columns,
        "close_present": "close" in df.columns,
    }

    if "date" in df.columns:
        checks["date_is_datetime"] = (
            pd.api.types.is_datetime64_any_dtype(df["date"])
        )

    if "close" in df.columns:
        checks["close_is_numeric"] = (
            pd.api.types.is_numeric_dtype(df["close"])
        )

    return checks

In [28]:
storage_checks = validate_spy_storage(spy_raw)

print(storage_checks)

{'not_empty': True, 'date_present': True, 'close_present': True, 'date_is_datetime': True, 'close_is_numeric': True}


In [29]:
display(spy_raw.head())

print("Shape:", spy_raw.shape)

print(
    "Date range:",
    spy_raw["date"].min(),
    "to",
    spy_raw["date"].max()
)

,date,close
0,2016-08-26,217.289993
1,2016-08-29,218.360001
2,2016-08-30,218.000000
3,2016-08-31,217.380005
4,2016-09-01,217.389999


Shape: (2512, 2)
Date range: 2016-08-26 00:00:00 to 2026-08-25 00:00:00
